# Synapse Notebook: PySpark - Reading ADLS & Creating Tables
**Language:** Python (PySpark)  
**Attach to:** Spark pool (e.g., sparkpool1)  
**Purpose:** Read data from ADLS, transform, and create tables in Synapse

## CELL 1: Initialize and set up storage connection

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType
import datetime

spark = SparkSession.builder.appName("SynapseDemo").getOrCreate()

# Display Spark version
print(f"Spark Version: {spark.version}")

## CELL 2: Set up ADLS connection credentials

In [ ]:
# Method 1: Using storage account key
spark.conf.set(
    "fs.azure.account.key.maybatchtrainingadls.dfs.core.windows.net",
    "your-access-key-or-sas-token"
)

# Method 2: Using managed identity (recommended in Synapse)
# spark.conf.set("fs.azure.auth.type", "OAuth")
# spark.conf.set("fs.azure.auth.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

adls_path = "abfss://maybarch-adlsgen2@maybatchtrainingadls.dfs.core.windows.net"
print(f"ADLS Path configured: {adls_path}")

## CELL 3: Read CSV file from ADLS

In [ ]:
df_customers_csv = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("delimiter", ",") \
    .load(f"{adls_path}/datasets/customers.csv")

# Display the data
df_customers_csv.show(10)
df_customers_csv.printSchema()

## CELL 4: Read Parquet file from ADLS

In [ ]:
df_customers_parquet = spark.read \
    .format("parquet") \
    .load(f"{adls_path}/transformed_data/customers/")

df_customers_parquet.show(5)
print(f"Parquet file row count: {df_customers_parquet.count()}")

## CELL 5: Read JSON file from ADLS

In [ ]:
df_customers_json = spark.read \
    .format("json") \
    .option("multiline", True) \
    .load(f"{adls_path}/output/SalesLT.CustomerAddress.json")

df_customers_json.show()
df_customers_json.printSchema()

## CELL 6: Create DataFrame with explicit schema

In [ ]:
schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("created_at", DateType(), True)
])

df_customers_explicit = spark.read \
    .format("csv") \
    .schema(schema) \
    .option("header", True) \
    .load(f"{adls_path}/datasets/customers.csv")

df_customers_explicit.show()
df_customers_explicit.printSchema()

## CELL 7: Data transformation and cleanup

In [ ]:
from pyspark.sql.functions import col, trim, upper, to_date, year, month

df_cleaned = df_customers_csv \
    .withColumn("customer_name", trim(upper(col("customer_name")))) \
    .withColumn("created_at", to_date(col("created_at"), "yyyy-MM-dd")) \
    .withColumn("year", year(col("created_at"))) \
    .withColumn("month", month(col("created_at"))) \
    .filter(col("customer_id").isNotNull())

df_cleaned.show()
print(f"Cleaned data: {df_cleaned.count()} rows")

## CELL 8: Create temporary view (session-scoped)

In [ ]:
df_cleaned.createOrReplaceTempView("vw_customers_temp")

# Query using SQL
result = spark.sql("SELECT * FROM vw_customers_temp WHERE year = 2022 LIMIT 10")
result.show()

## CELL 9: Create external table (permanent, stored in Synapse)

In [ ]:
df_cleaned.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", f"{adls_path}/transformed_data/customers_external/") \
    .saveAsTable("customers_external")

print("External table 'customers_external' created")

## CELL 10: Create managed table (data stored in Synapse warehouse)

# Create managed table - Data will be stored in Synapse default warehouse location
# Location: synapse/workspaces/{workspace-name}/warehouse/customers_managed/
# Note: No explicit path specified → Synapse manages the ADLS location automatically
# Format: Delta Lake (provides ACID transactions and time-travel capabilities)

In [ ]:

df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customers_managed")

print("Managed table 'customers_managed' created")
print("Data stored in: synapse/workspaces/{workspace-name}/warehouse/customers_managed/")

In [ ]:
df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customers_managed")

print("Managed table 'customers_managed' created")

## CELL 11: Register table and query it

In [ ]:
result = spark.sql("SELECT COUNT(*) as TotalCustomers FROM customers_managed")
result.show()

# Verify table exists
spark.sql("SHOW TABLES").filter("tableName LIKE '%customers%'").show()

## CELL 12: Aggregation operations

In [ ]:
df_aggregated = df_cleaned.groupBy("year", "month") \
    .agg({
        "customer_id": "count"
    }) \
    .withColumnRenamed("count(customer_id)", "TotalCustomers") \
    .orderBy("year", "month")

df_aggregated.show()

## CELL 13: Window functions

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, sum as spark_sum

window_spec = Window.partitionBy("customer_id").orderBy(col("created_at"))

df_with_window = df_cleaned.select("customer_id", "customer_name", "email", "created_at").withColumn(
    "RunningCount",
    spark_sum(col("customer_id")).over(window_spec)
).withColumn(
    "PrevCustomerName",
    lag(col("customer_name")).over(window_spec)
).withColumn(
    "NextCustomerCreatedDate",
    lead(col("created_at")).over(window_spec)
)

df_with_window.show()

## CELL 14: Write to ADLS in different formats

In [ ]:
# Write as Parquet (columnar, compressed)
df_cleaned.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("year") \
    .save(f"{adls_path}/transformed_data/customers_parquet_partitioned/")

print("Parquet written with year partition")

# Write as Delta Lake (with ACID transactions)
df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .save(f"{adls_path}/transformed_data/customers_delta/")

print("Delta Lake written with year partition")

## CELL 15: Repartition data for optimization

In [ ]:
df_repartitioned = df_cleaned.repartition(4, col("year"))
df_repartitioned.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(f"{adls_path}/transformed_data/customers_repartitioned/")

print(f"Data repartitioned into 4 partitions and written to ADLS")
print(f"Number of output files: {df_repartitioned.rdd.getNumPartitions()}")

## CELL 16: Read back and verify

In [ ]:
df_verify = spark.read \
    .format("parquet") \
    .load(f"{adls_path}/transformed_data/customers_parquet_partitioned/")

print(f"Total rows in verified data: {df_verify.count()}")
df_verify.show(5)
df_verify.printSchema()

## BONUS: Performance tips and best practices

In [ ]:
# Enable caching for frequently used DataFrames
df_cleaned.cache()
print(f"DataFrame cached. Memory usage info:")
df_cleaned.storageLevel

# Check query plan
df_cleaned.filter(col("SalesAmount") > 1000).explain(extended=True)

# Unpersist when done
# df_cleaned.unpersist()

## CLEANUP: Remove All Created Tables & Data

**Purpose:** Clean up all tables, views, and ADLS data created during this notebook execution

**Why this matters:**
- Prevents accidental data duplication on re-runs
- Avoids unnecessary ADLS storage costs
- Ensures idempotent notebook behavior (safe to run multiple times)
- Cleans up both Synapse metadata and underlying ADLS files

**What gets cleaned:**
- ✓ Managed tables (customers_managed)
- ✓ External tables (customers_external)
- ✓ Temporary views (vw_customers_temp)
- ✓ All transformed data in ADLS (/transformed_data/)

In [ ]:
import os

# ============================================================================
# STEP 1: Drop Temporary Views
# ============================================================================
try:
    spark.sql("DROP VIEW IF EXISTS vw_customers_temp")
    print("✓ Dropped temporary view: vw_customers_temp")
except Exception as e:
    print(f"ℹ Temporary view not found or already dropped: {e}")

# ============================================================================
# STEP 2: Drop Managed Tables (data in warehouse location gets deleted)
# ============================================================================
try:
    spark.sql("DROP TABLE IF EXISTS customers_managed")
    print("✓ Dropped managed table: customers_managed")
    print("  → Data removed from: synapse/workspaces/{workspace-name}/warehouse/customers_managed/")
except Exception as e:
    print(f"ℹ Managed table not found or already dropped: {e}")

# ============================================================================
# STEP 3: Drop External Tables (only metadata removed, ADLS data deleted separately)
# ============================================================================
try:
    spark.sql("DROP TABLE IF EXISTS customers_external")
    print("✓ Dropped external table: customers_external")
except Exception as e:
    print(f"ℹ External table not found or already dropped: {e}")

# ============================================================================
# STEP 4: Remove All Transformed Data from ADLS
# ============================================================================
def delete_adls_path(spark, path):
    """Helper function to delete ADLS paths"""
    try:
        dbutils.fs.rm(path, recurse=True)
        return True
    except Exception as e:
        return False

# List of ADLS paths to clean up
paths_to_clean = [
    f"{adls_path}/transformed_data/customers_external/",
    f"{adls_path}/transformed_data/customers_parquet_partitioned/",
    f"{adls_path}/transformed_data/customers_delta/",
    f"{adls_path}/transformed_data/customers_repartitioned/"
]

print("\nCleaning up ADLS paths:")
for path in paths_to_clean:
    if delete_adls_path(spark, path):
        print(f"✓ Deleted: {path}")
    else:
        print(f"ℹ Path not found or already deleted: {path}")

# ============================================================================
# STEP 5: Verify Cleanup - Show remaining tables
# ============================================================================
print("\n" + "="*70)
print("CLEANUP VERIFICATION")
print("="*70)

remaining_tables = spark.sql("SHOW TABLES").filter("tableName LIKE '%customers%'")
if remaining_tables.count() == 0:
    print("✓ All customer tables successfully removed")
else:
    print(f"⚠ {remaining_tables.count()} customer table(s) still remain:")
    remaining_tables.show()

# Clear cached DataFrames
print("\nClearing cached DataFrames...")
spark.catalog.clearCache()
print("✓ All cached data cleared")

print("\n" + "="*70)
print("CLEANUP COMPLETE - Notebook is ready for fresh run")
print("="*70)